# Late Transaction Handling & Historical Revenue Correction

**Tools:** Databricks | PySpark | Auto Loader | Delta Lake

## Business problem
A transaction can happen on one day but reach the data platform later.
If the daily revenue report has already been created, that historical
number can become incomplete.

This notebook captures the data, cleans and validates it, creates a daily
revenue report, detects late-arriving transactions, identifies affected
dates, recalculates those dates from trusted Silver data, and updates
Gold with Delta MERGE.

**Key idea:** correct only the dates that changed instead of rebuilding
the complete historical revenue table.

## Step 0 — Project paths

Keeping raw input, processing layers, checkpoints, schema information,
and rejected records separate makes the pipeline easier to maintain.

In [0]:
base_path = "/Volumes/workspace/default/spark_data/late_txn_project"

raw_path = f"{base_path}/sat_act"
bronze_path = f"{base_path}/bronze/sales"
silver_path = f"{base_path}/silver/sales"
gold_path = f"{base_path}/gold/revenue"
checkpoint_path = f"{base_path}/checkpoints/bronze"
schema_path = f"{base_path}/schema/bronze"
quarantine_path = f"{base_path}/error/quarantine"

print("Project input folder:", raw_path)

Project input folder: /Volumes/workspace/default/spark_data/late_txn_project/sat_act


## Step 0b — Confirm the input file
Upload `sales_2000_rows.csv` to the `sat_act` folder before running.

In [0]:
dbutils.fs.ls(raw_path)

[FileInfo(path='dbfs:/Volumes/workspace/default/spark_data/late_txn_project/sat_act/sales_2000_rows.csv', name='sales_2000_rows.csv', size=70585, modificationTime=1785869909000)]

## Step 1 — Understand the incoming data

Before building the pipeline, I first inspect the source data. In
particular, `txn_date` tells me when the transaction happened and
`ingestion_date` tells me when the platform received it.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

source_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(raw_path)
)

source_df.show(10, truncate=False)
source_df.printSchema()
print("Total input records:", source_df.count())

+------+-------+----------+------+--------------+
|txn_id|user_id|txn_date  |amount|ingestion_date|
+------+-------+----------+------+--------------+
|1     |427    |2024-01-08|2353  |2024-01-08    |
|2     |225    |2024-01-15|939   |2024-01-15    |
|3     |446    |2024-02-17|812   |2024-02-27    |
|4     |402    |2024-01-28|344   |2024-01-28    |
|5     |147    |2024-01-14|4239  |2024-01-15    |
|6     |408    |2024-01-02|1728  |2024-01-12    |
|7     |466    |2024-02-11|3536  |2024-02-21    |
|8     |212    |2024-01-29|2378  |2024-02-13    |
|9     |103    |2024-02-18|3562  |2024-02-18    |
|10    |274    |2024-01-18|1863  |2024-01-18    |
+------+-------+----------+------+--------------+
only showing top 10 rows
root
 |-- txn_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- txn_date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- ingestion_date: string (nullable = true)

Total input records: 2000


### Quick data-quality profile

In [0]:
source_quality = source_df.select(
    F.count("*").alias("total_records"),
    F.countDistinct("txn_id").alias("unique_txn_ids"),
    F.sum(F.when(F.col("txn_id").isNull(), 1).otherwise(0)).alias("null_txn_ids"),
    F.sum(
        F.when(F.col("amount").cast("double") <= 0, 1).otherwise(0)
    ).alias("non_positive_amounts")
)

display(source_quality)

total_records,unique_txn_ids,null_txn_ids,non_positive_amounts
2000,2000,0,0


## Step 2 — Bronze Layer: capture what arrived

Bronze stays close to the source. I avoid business transformations here
and keep ingestion metadata for traceability. Auto Loader allows new
files to be picked up without changing the ingestion logic.

In [0]:
bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(raw_path)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
(
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(bronze_path)
    .awaitTermination()
)

print("Bronze ingestion completed.")

Bronze ingestion completed.


In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

print("Bronze records:", bronze_df.count())
display(bronze_df.limit(10))

Bronze records: 2000


txn_id,user_id,txn_date,amount,ingestion_date,_rescued_data,_ingested_at
1,427,2024-01-08,2353,2024-01-08,null,2026-08-04T19:07:49.836Z
2,225,2024-01-15,939,2024-01-15,null,2026-08-04T19:07:49.836Z
3,446,2024-02-17,812,2024-02-27,null,2026-08-04T19:07:49.836Z
4,402,2024-01-28,344,2024-01-28,null,2026-08-04T19:07:49.836Z
5,147,2024-01-14,4239,2024-01-15,null,2026-08-04T19:07:49.836Z
6,408,2024-01-02,1728,2024-01-12,null,2026-08-04T19:07:49.836Z
7,466,2024-02-11,3536,2024-02-21,null,2026-08-04T19:07:49.836Z
8,212,2024-01-29,2378,2024-02-13,null,2026-08-04T19:07:49.836Z
9,103,2024-02-18,3562,2024-02-18,null,2026-08-04T19:07:49.836Z
10,274,2024-01-18,1863,2024-01-18,null,2026-08-04T19:07:49.836Z


## Step 3 — Silver Layer: make the data trustworthy

Here I standardize data types, separate invalid records, and remove
duplicate transaction IDs. If the same transaction appears more than
once, I keep the latest ingested version.

In [0]:
typed_df = (
    bronze_df
    .withColumn("txn_date", F.to_date("txn_date"))
    .withColumn("ingestion_date", F.to_date("ingestion_date"))
    .withColumn("amount", F.col("amount").cast("double"))
)

invalid_df = typed_df.filter(
    F.col("txn_id").isNull()
    | F.col("txn_date").isNull()
    | F.col("ingestion_date").isNull()
    | F.col("amount").isNull()
    | (F.col("amount") <= 0)
)

valid_df = typed_df.filter(
    F.col("txn_id").isNotNull()
    & F.col("txn_date").isNotNull()
    & F.col("ingestion_date").isNotNull()
    & F.col("amount").isNotNull()
    & (F.col("amount") > 0)
)

In [0]:
(
    invalid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(quarantine_path)
)

dedup_window = (
    Window
    .partitionBy("txn_id")
    .orderBy(F.col("ingestion_date").desc(), F.col("_ingested_at").desc())
)

silver_df = (
    valid_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path)
)

print("Silver records:", silver_df.count())
print("Quarantined records:", invalid_df.count())
display(silver_df.limit(10))

Silver records: 2000
Quarantined records: 0


txn_id,user_id,txn_date,amount,ingestion_date,_rescued_data,_ingested_at
1,427,2024-01-08,2353.0,2024-01-08,null,2026-08-04T19:07:49.836Z
10,274,2024-01-18,1863.0,2024-01-18,null,2026-08-04T19:07:49.836Z
100,441,2024-02-22,2552.0,2024-03-03,null,2026-08-04T19:07:49.836Z
1000,203,2024-01-09,992.0,2024-01-24,null,2026-08-04T19:07:49.836Z
1001,179,2024-01-07,3782.0,2024-01-07,null,2026-08-04T19:07:49.836Z
1002,338,2024-01-21,1089.0,2024-01-26,null,2026-08-04T19:07:49.836Z
1003,374,2024-01-23,3802.0,2024-01-24,null,2026-08-04T19:07:49.836Z
1004,257,2024-01-30,1082.0,2024-02-01,null,2026-08-04T19:07:49.836Z
1005,146,2024-01-11,430.0,2024-01-13,null,2026-08-04T19:07:49.836Z
1006,210,2024-02-23,1315.0,2024-02-26,null,2026-08-04T19:07:49.836Z


## Step 4 — Initial Gold report

To make the late-arrival correction visible with the supplied historical
dataset, the first report contains transactions received on their
transaction date. The late records are then treated as arriving later.
This gives a clear before → correction → after flow.

In [0]:
on_time_df = silver_df.filter(
    F.col("ingestion_date") <= F.col("txn_date")
)

gold_initial_df = (
    on_time_df
    .groupBy("txn_date")
    .agg(
        F.round(F.sum("amount"), 2).alias("daily_revenue")
    )
)

(
    gold_initial_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("txn_date")
    .save(gold_path)
)

print("Initial Gold report created.")
display(
    spark.read.format("delta").load(gold_path).orderBy("txn_date")
)

Initial Gold report created.


txn_date,daily_revenue
2024-01-01,11523.0
2024-01-02,10277.0
2024-01-03,35612.0
2024-01-04,16203.0
2024-01-05,21897.0
2024-01-06,32944.0
2024-01-07,39752.0
2024-01-08,26580.0
2024-01-09,9725.0
2024-01-10,25370.0


## Step 5 — Detect late transactions

A transaction is late when `ingestion_date > txn_date`. I also calculate
the delay in days so the result is easier to interpret.

In [0]:
late_df = (
    silver_df
    .filter(F.col("ingestion_date") > F.col("txn_date"))
    .withColumn(
        "delay_days",
        F.datediff("ingestion_date", "txn_date")
    )
)

print("Late transactions:", late_df.count())

display(
    late_df
    .select(
        "txn_id",
        "txn_date",
        "ingestion_date",
        "delay_days",
        "amount"
    )
    .orderBy(F.desc("delay_days"))
    .limit(20)
)

Late transactions: 1415


txn_id,txn_date,ingestion_date,delay_days,amount
1196,2024-02-03,2024-02-18,15,4393.0
1155,2024-02-08,2024-02-23,15,2415.0
1127,2024-01-27,2024-02-11,15,831.0
1242,2024-02-03,2024-02-18,15,880.0
1152,2024-02-03,2024-02-18,15,3353.0
1175,2024-02-25,2024-03-11,15,4853.0
1036,2024-02-19,2024-03-05,15,2006.0
1086,2024-01-19,2024-02-03,15,4034.0
1234,2024-01-27,2024-02-11,15,4054.0
1148,2024-01-17,2024-02-01,15,2602.0


### Late-arrival impact

In [0]:
late_summary = late_df.agg(
    F.count("*").alias("late_transactions"),
    F.countDistinct("txn_date").alias("affected_dates"),
    F.round(F.avg("delay_days"), 2).alias("average_delay_days"),
    F.max("delay_days").alias("maximum_delay_days"),
    F.round(F.sum("amount"), 2).alias("late_transaction_value")
)

display(late_summary)

late_transactions,affected_dates,average_delay_days,maximum_delay_days,late_transaction_value
1415,60,6.32,15,3708520.0


## Step 6 — Find the affected historical dates

I do not want to rebuild every historical revenue value just because
some transactions arrived late. First I identify the distinct dates
that actually need correction.

In [0]:
affected_dates = late_df.select("txn_date").distinct()

print(
    "Historical dates requiring correction:",
    affected_dates.count()
)

display(affected_dates.orderBy("txn_date"))

Historical dates requiring correction: 60


txn_date
2024-01-01
2024-01-02
2024-01-03
2024-01-04
2024-01-05
2024-01-06
2024-01-07
2024-01-08
2024-01-09
2024-01-10


## Step 7 — Recalculate only those dates

The corrected revenue is calculated from the complete cleaned Silver
dataset, not only from the late records. This gives the true total for
each affected date.

In [0]:
recomputed_df = (
    silver_df
    .join(affected_dates, on="txn_date", how="inner")
    .groupBy("txn_date")
    .agg(
        F.round(F.sum("amount"), 2).alias("daily_revenue")
    )
)

display(recomputed_df.orderBy("txn_date"))

txn_date,daily_revenue
2024-01-01,56799.0
2024-01-02,66741.0
2024-01-03,93732.0
2024-01-04,70961.0
2024-01-05,69963.0
2024-01-06,94410.0
2024-01-07,95732.0
2024-01-08,108769.0
2024-01-09,88910.0
2024-01-10,69564.0


## Step 8 — Save the original values for comparison

I keep the original Gold values for the affected dates so I can verify
the correction after the MERGE.

In [0]:
gold_before_df = (
    spark.read
    .format("delta")
    .load(gold_path)
    .join(affected_dates, on="txn_date", how="inner")
    .withColumnRenamed("daily_revenue", "revenue_before")
)

## Step 9 — Delta MERGE

Only affected dates are sent to the MERGE:

- existing date → update
- new date → insert
- unaffected date → untouched

In [0]:
from delta.tables import DeltaTable

gold_table = DeltaTable.forPath(spark, gold_path)

(
    gold_table.alias("target")
    .merge(
        recomputed_df.alias("source"),
        "target.txn_date = source.txn_date"
    )
    .whenMatchedUpdate(
        set={"daily_revenue": "source.daily_revenue"}
    )
    .whenNotMatchedInsert(
        values={
            "txn_date": "source.txn_date",
            "daily_revenue": "source.daily_revenue"
        }
    )
    .execute()
)

print("Gold correction completed.")

Gold correction completed.


## Step 10 — Prove the correction worked

Instead of only printing a success message, I compare the revenue
before and after the correction.

In [0]:
gold_after_df = (
    spark.read
    .format("delta")
    .load(gold_path)
    .join(affected_dates, on="txn_date", how="inner")
    .withColumnRenamed("daily_revenue", "revenue_after")
)

comparison_df = (
    gold_before_df
    .join(gold_after_df, on="txn_date", how="inner")
    .withColumn(
        "revenue_change",
        F.round(
            F.col("revenue_after") - F.col("revenue_before"),
            2
        )
    )
    .orderBy(F.desc("revenue_change"))
)

display(comparison_df)

txn_date,revenue_before,revenue_after,revenue_change
2024-02-20,75630.0,75630.0,0.0
2024-02-08,88308.0,88308.0,0.0
2024-02-07,83162.0,83162.0,0.0
2024-02-16,123501.0,123501.0,0.0
2024-02-27,70211.0,70211.0,0.0
2024-02-17,97372.0,97372.0,0.0
2024-02-05,98618.0,98618.0,0.0
2024-02-09,83724.0,83724.0,0.0
2024-02-14,82122.0,82122.0,0.0
2024-02-18,61959.0,61959.0,0.0


## Step 11 — Final data-quality checks

In [0]:
final_silver_df = spark.read.format("delta").load(silver_path)

null_txn_id = final_silver_df.filter(F.col("txn_id").isNull()).count()

duplicate_txn_id_groups = (
    final_silver_df
    .groupBy("txn_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_amounts = final_silver_df.filter(
    F.col("amount") <= 0
).count()

print("FINAL DATA QUALITY")
print("-" * 35)
print("Null txn_id:", null_txn_id)
print("Duplicate txn_id groups:", duplicate_txn_id_groups)
print("Invalid amount rows:", invalid_amounts)

FINAL DATA QUALITY
-----------------------------------
Null txn_id: 0
Duplicate txn_id groups: 0
Invalid amount rows: 0


## Step 12 — Processing control

This control table records the latest processed transaction date and
pipeline run timestamp. It provides an audit point and can support
incremental processing in a scheduled production implementation.

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.watermark_control (
    table_name STRING,
    last_processed_date DATE,
    last_run_timestamp TIMESTAMP
)
""")

max_date = (
    final_silver_df
    .agg(F.max("txn_date").alias("max_date"))
    .first()["max_date"]
)

watermark_df = (
    spark.createDataFrame(
        [("gold_revenue", max_date)],
        ["table_name", "last_processed_date"]
    )
    .withColumn("last_run_timestamp", F.current_timestamp())
)

watermark_df.createOrReplaceTempView("current_watermark")

spark.sql("""
MERGE INTO workspace.default.watermark_control AS target
USING current_watermark AS source
ON target.table_name = source.table_name
WHEN MATCHED THEN UPDATE SET
    target.last_processed_date = source.last_processed_date,
    target.last_run_timestamp = source.last_run_timestamp
WHEN NOT MATCHED THEN INSERT (
    table_name,
    last_processed_date,
    last_run_timestamp
)
VALUES (
    source.table_name,
    source.last_processed_date,
    source.last_run_timestamp
)
""")

display(spark.table("workspace.default.watermark_control"))

table_name,last_processed_date,last_run_timestamp
gold_revenue,2024-02-29,2026-08-07T16:26:56.861Z


## Final project summary

The pipeline now:

1. captures source files in Bronze,
2. cleans and validates records in Silver,
3. creates an initial daily revenue report,
4. detects late-arriving transactions,
5. identifies affected historical dates,
6. recalculates those dates from complete Silver data,
7. updates Gold with Delta MERGE, and
8. verifies the correction with a before/after comparison.

**Business value:** accurate historical reporting without unnecessarily
rebuilding unaffected dates.